# 🎨 ChromaDB Tutorial — Explained

**What this notebook does:** walks through ChromaDB, an open-source vector database, end to end — embedding text, creating collections, adding and querying documents (both by text and by raw vectors), persistent storage, metadata filtering, and upsert/delete operations. Each cell below has a short explanation directly above it.

### 📦 Cell 0 — Install the libraries

Installs **ChromaDB** (the vector database itself) and **sentence-transformers** (the library used to load an embedding model). These two are the only real dependencies this whole notebook needs.

In [ ]:
!pip install chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found ex

### 🧰 Cell 1 — Import the tools, and unlock a safety switch

Imports the ChromaDB client and `SentenceTransformer`. `ALLOW_RESET=True` matters specifically for Cell 12 later — ChromaDB blocks its destructive `reset()` call by default, and this environment variable is what allows it to run at all.

In [ ]:
import chromadb
import os
os.environ['ALLOW_RESET'] = "True"
from sentence_transformers import SentenceTransformer

### 🔤 Cell 2 — Turn sentences into embeddings, and compare them

Loads a small, fast embedding model (`all-MiniLM-L6-v2`) and turns 3 plain sentences into embeddings with `model.encode()`. `model.similarity()` then compares every embedding against every other one — this is dense vector retrieval's core math: same model on both sides, then compare by similarity (see *Introduction to Vector Database* Q1–Q2 and the RAG workflow notes).

In [ ]:
# load a embeddings model from SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)

# Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(3, 384)
tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])


### 🖨️ Cell 3 — Print the similarity scores readably

Loops through every sentence pair and prints its similarity score to 4 decimal places, so Cell 2's raw similarity matrix is easy to read at a glance instead of staring at a grid of numbers.

In [ ]:
# Formatting the output for a clear view
for idx_i, sentence1 in enumerate(sentences):
    print(sentence1)
    for idx_j, sentence2 in enumerate(sentences):
        print(f" - {sentence2: <30}: {similarities[idx_i][idx_j]:.4f}")

The weather is lovely today.
 - The weather is lovely today.  : 1.0000
 - It's so sunny outside!        : 0.6660
 - He drove to the stadium.      : 0.1046
It's so sunny outside!
 - The weather is lovely today.  : 0.6660
 - It's so sunny outside!        : 1.0000
 - He drove to the stadium.      : 0.1411
He drove to the stadium.
 - The weather is lovely today.  : 0.1046
 - It's so sunny outside!        : 0.1411
 - He drove to the stadium.      : 1.0000


### 🗄️ Cell 4 — Start an in-memory ChromaDB client

`chromadb.Client()` creates a temporary, **in-memory** database — nothing is saved to disk, so everything resets the moment the notebook restarts. (Cell 10 later switches to a version that *does* persist to disk.)

In [ ]:
chroma_client = chromadb.Client() # load the chromadb client

### 📁 Cell 5 — Create a collection

A **collection** in ChromaDB is like a table in a normal database — a named container that holds documents and their embeddings together. This creates one called `my_collection`.

In [ ]:
collection = chroma_client.create_collection(name="my_collection") # create a collection that will store vectors

### ➕ Cell 6 — Add documents to the collection

Adds two plain-text documents with unique `ids`. Notice there's no explicit embedding step here at all — ChromaDB automatically embeds the text behind the scenes using its own default embedding model.

In [ ]:
# Add data to collection
collection.add(
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges"
    ],
    ids=["id1", "id2"]
)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 29.9MiB/s]


### 🔍 Cell 7 — Query the collection

Runs an actual semantic search: Chroma embeds the query text automatically (same as Cell 6), then returns the `n_results` most similar documents — dense vector retrieval end-to-end, in two lines.

In [ ]:
# Querying on the vectordb
results = collection.query(
    query_texts=["This is a query document about hawaii"], # Chroma will embed this for you
    n_results=2 # how many results to return
)

print(results)

{'ids': [['id1', 'id2']], 'embeddings': None, 'documents': [['This is a document about pineapple', 'This is a document about oranges']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None]], 'distances': [[1.0404013395309448, 1.2430801391601562]]}


### 👀 Cell 8 — Peek at the raw embeddings

By default Chroma hides the embedding vectors in its output, since they're long and not usually useful to look at. Passing `include=['embeddings']` forces it to show them anyway, so you can see the actual numbers being compared under the hood.

In [ ]:
# The embeddings are included in the output because it is a very big vector.
collection.get(include=['embeddings'])

{'ids': ['id1', 'id2'],
 'embeddings': array([[-7.09366705e-03,  6.55461624e-02, -1.16605619e-02,
          8.64458233e-02, -3.95435169e-02,  5.80052435e-02,
         -2.40756441e-02,  1.49505902e-02,  9.91733465e-03,
         -2.10740846e-02,  8.10110569e-02,  3.47032249e-02,
         -2.12537181e-02, -1.33663500e-02, -3.32268663e-02,
          2.06374303e-02,  3.50654521e-03,  3.84415500e-02,
         -1.53143816e-02,  7.31229112e-02,  5.14781438e-02,
          1.06985062e-01, -4.18041795e-02,  3.10128443e-02,
         -1.11353695e-02, -1.37997670e-02, -2.21654493e-02,
         -6.40951097e-02, -2.69082189e-02, -1.26517490e-02,
         -4.96093649e-03,  5.54248430e-02,  1.17435031e-01,
          5.17346859e-02, -5.17162047e-02, -3.09536438e-02,
          1.01460829e-01, -6.95117563e-02,  1.60278112e-01,
          4.23167050e-02, -2.92910207e-02, -6.58734655e-03,
          4.21533249e-02,  1.74002461e-02,  1.13289645e-02,
          5.93366623e-02, -6.98440224e-02,  5.52196614e-03,
  

### Persistent storage

### 💾 Cell 10 — Switch to a persistent client

Unlike Cell 4's in-memory client, `PersistentClient` saves everything to a folder on disk (`/content/my_chroma_db`), so the data survives a restart — this is what a real application would actually use.

In [ ]:
# Data will be persisted automatically and loaded on start (if it exists).
client = chromadb.PersistentClient(path="/content/my_chroma_db")

### ❤️ Cell 11 — Check the connection is alive

`heartbeat()` just returns a timestamp-like number to confirm the client is connected — a simple health check, nothing more.

In [ ]:
client.heartbeat() # returns a nanosecond heartbeat. Useful for making sure the client remains connected.

1783336092189833629

### ⚠️ Cell 12 — Wipe the entire database

`reset()` deletes **everything** — every collection, document, and embedding — permanently. This is exactly why Cell 1 had to set `ALLOW_RESET=True` first; Chroma disables this by default to prevent accidental data loss.

In [ ]:
client.reset() # Empties and completely resets the database. ⚠️ This is destructive and not reversible.

True

### 🧩 Cell 13 — Define an explicit embedding function

Instead of letting Chroma pick its own default embedding model silently, this wraps the *same* `all-MiniLM-L6-v2` model from Cell 2 into a format Chroma understands — making it explicit which model does the embedding, rather than leaving it implicit.

In [ ]:
from chromadb.utils import embedding_functions
sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### 📁 Cell 14 — Create a collection using that specific model

`get_or_create_collection` creates the collection if it doesn't exist yet, or just fetches it if it does — safer than `create_collection`, which errors on a duplicate name. Passing `embedding_function` here means every document added to *this* collection is embedded by that exact model, every time.

In [ ]:
# To pass an embedding function to the vectordb use embedding_function
collection = client.get_or_create_collection(name="my_collection_2", embedding_function=sentence_transformer_ef)

### 👀 Cell 15 — Look at the first few items

`peek()` is a quick sanity check — it returns (up to) the first 10 items stored in the collection, useful for confirming things actually got added.

In [ ]:
print(collection.peek()) # returns a list of the first 10 items in the collection

{'ids': [], 'embeddings': array([], dtype=float64), 'documents': [], 'uris': None, 'included': ['metadatas', 'documents', 'embeddings'], 'data': None, 'metadatas': []}


### 🔢 Cell 16 — Count how many items are stored

Returns a single number: the total count of documents currently sitting in the collection.

In [ ]:
print(collection.count()) # returns the number of items in the collection

0


### ✏️ Cell 17 — Rename the collection

Changes the collection's name in place — to `vect-db` here — while everything else about it (its documents, embeddings, metadata) stays exactly the same.

In [ ]:
print(collection.modify(name="vect-db")) # Rename the collection

None


### 📐 Cell 18 — Choose the similarity math

`metadata={'hnsw:space': 'cosine'}` tells Chroma's HNSW index (the same approximate-search structure from the vector database notes) to measure closeness using **cosine similarity** instead of its default, L2 (straight-line) distance — changing what “closest” actually means for this collection.

In [ ]:
# create_collection also takes an optional metadata argument which can be used to customize the distance method
# of the embedding space by setting the value of hnsw:space
collection = client.create_collection(
        name="KnowledgeBase",
        metadata={"hnsw:space": "cosine"} # l2 is the default
    )

### ➕ Cell 19 — Add documents with metadata attached

Each document now carries extra structured tags (`chapter`, `verse`) alongside its text and id — metadata that can later be used to *filter* searches on top of semantic similarity.

In [ ]:
collection.add(
    documents=["lorem ipsum...", "doc2", "doc3",],
    metadatas=[{"chapter": "3", "verse": "16"}, {"chapter": "3", "verse": "5"}, {"chapter": "29", "verse": "11"}],
    ids=["id1", "id2", "id3"]
)

### 👀 Cell 20 — Confirm the documents were added

Same `peek()` as Cell 15, just checking that all 3 documents (and their metadata) landed correctly this time.

In [ ]:
collection.peek()

{'ids': ['id1', 'id2', 'id3'],
 'embeddings': array([[ 0.03684678,  0.02051685,  0.07883844, ..., -0.10914499,
          0.02406928, -0.00106998],
        [-0.05917219,  0.03088627,  0.05787699, ...,  0.0756631 ,
          0.06910367,  0.04536711],
        [-0.07160491,  0.01184765,  0.01126068, ...,  0.04132251,
          0.06954484,  0.0496698 ]]),
 'documents': ['lorem ipsum...', 'doc2', 'doc3'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'chapter': '3', 'verse': '16'},
  {'chapter': '3', 'verse': '5'},
  {'verse': '11', 'chapter': '29'}]}

### 🔢 Cell 21 — Confirm the count is 3

Same idea as Cell 16 — a quick count to double-check nothing was dropped.

In [ ]:
collection.count() # count no. of documents added

3

You can also store documents elsewhere, and just supply a list of embeddings and metadata to Chroma. You can use the ids to associate the embeddings with your documents stored elsewhere.



### 📁 Cell 23 — A fresh collection for the next example

Same pattern as Cell 14 — a new, empty collection using the explicit embedding function — set up specifically to demonstrate supplying raw embeddings directly, next.

In [ ]:
collection = client.create_collection(name="Test1", embedding_function=sentence_transformer_ef)

### ➕ Cell 24 — Add raw embedding vectors directly, no text at all

Instead of handing Chroma text to embed itself (like Cell 6 or 19), this passes already-made numeric vectors straight in. This is the “store your documents elsewhere, just supply the vector + metadata” pattern the note above mentions — useful when your actual documents live in a separate system.

In [ ]:
collection.add(
    embeddings=[[1.1, 2.3, 3.2], [4.5, 6.9, 4.4], [1.1, 2.3, 3.2]],
    metadatas=[{"chapter": "3", "verse": "16"}, {"chapter": "3", "verse": "5"}, {"chapter": "29", "verse": "11"}],
    ids=["id1", "id2", "id3"]
)

### 🔍 Cell 25 — Query using a raw vector instead of text

Since this collection's items were added as raw vectors, the query is a raw vector too (`query_embeddings`) rather than text — Chroma runs nearest-neighbor search directly on the numbers, with no embedding step needed on either side this time.

In [ ]:
collection.query(
    query_embeddings=[[11.1, 12.1, 13.1]],
    n_results=10,
)

{'ids': [['id2', 'id1', 'id3']],
 'embeddings': None,
 'documents': [[None, None, None]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'verse': '5', 'chapter': '3'},
   {'chapter': '3', 'verse': '16'},
   {'chapter': '29', 'verse': '11'}]],
 'distances': [[0.02596461772918701,
   0.04634404182434082,
   0.04634404182434082]]}

### 🔎 Cell 26 — Fetch specific items by id, with an optional filter

`get()` retrieves exact items by their `id` — this is a direct lookup, not a similarity search. The `where` clause here filters by a `style` field that was never actually set on any item in Cell 24, so in practice this would return nothing.

In [ ]:
# items can also be retrieved from a collection using get

collection.get(
	ids=["id1", "id2", "id3"],
	where={"style": "style1"}
)

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

### 🔍 Cell 27 — Combine vector search with a metadata filter

Runs the same nearest-neighbor search as Cell 25, but adds `where={'chapter': '3'}` — Chroma first narrows the candidates down to only documents matching that metadata, then ranks the survivors by vector similarity. This is hybrid filtering: semantic search *and* exact structured filtering, together.

In [ ]:
collection.query(
    query_embeddings=[[11.1, 12.1, 13.1]],
    n_results=10,
    where={"chapter": "3"}
)

{'ids': [['id2', 'id1']],
 'embeddings': None,
 'documents': [[None, None]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'verse': '5', 'chapter': '3'},
   {'verse': '16', 'chapter': '3'}]],
 'distances': [[0.02596461772918701, 0.04634404182434082]]}

### 🎛️ Cell 28 — The filter operators available for `where`

The comment lists Chroma's comparison operators (`$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`). The code then demonstrates two of them: an exact match (`$eq`) and a “matches any of these values” filter (`$in`) — both narrowing the vector search the same way Cell 27 did.

In [ ]:
# $eq - equal to (string, int, float)
# $ne - not equal to (string, int, float)
# $gt - greater than (int, float)
# $gte - greater than or equal to (int, float)
# $lt - less than (int, float)
# $lte - less than or equal to (int, float)

print(collection.query(
    query_embeddings=[[11.1, 12.1, 13.1]],
    n_results=10,
    where={"chapter": {"$eq":"3"}}
))

print(collection.query(
    query_embeddings=[[11.1, 12.1, 13.1]],
    n_results=10,
    where={"chapter": {"$in":["3","29"]}}
))

{'ids': [['id2', 'id1']], 'embeddings': None, 'documents': [[None, None]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'chapter': '3', 'verse': '5'}, {'chapter': '3', 'verse': '16'}]], 'distances': [[0.02596461772918701, 0.04634404182434082]]}
{'ids': [['id2', 'id1', 'id3']], 'embeddings': None, 'documents': [[None, None, None]], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'chapter': '3', 'verse': '5'}, {'verse': '16', 'chapter': '3'}, {'chapter': '29', 'verse': '11'}]], 'distances': [[0.02596461772918701, 0.04634404182434082, 0.04634404182434082]]}


### 🔁 Cell 29 — Update-or-insert in one call

`upsert()` checks each `id`: if it already exists, it overwrites that item's data; if it doesn't exist yet, it creates it fresh — convenient when you don't want to check existence yourself before writing.

In [ ]:
# Chroma also supports an upsert operation, which updates existing items, or adds them if they don't yet exist.

collection.upsert(
    ids=["id1", "id2", "id3"],
    embeddings=[[1.1, 2.3, 3.2], [4.5, 6.9, 4.4], [1.1, 2.3, 3.2],],
    metadatas=[{"chapter": "3", "verse": "16"}, {"chapter": "3", "verse": "5"}, {"chapter": "29", "verse": "11"}],
    documents=["doc1", "doc2", "doc3",],
)

### 🗑️ Cell 30 — Delete by id AND metadata filter, together

`delete()` here only removes items matching **both** the given `ids` *and* the `where` filter. The `get()` call right after immediately checks what's left, confirming whether the delete actually matched anything.

In [ ]:
# to delete certain items using filters and ids

print(collection.delete(
    ids=["id1", "id2", "id3"],
	where={"chapter": "20"}
))

print(collection.get(
	ids=["id1", "id2", "id3"]
))

{'deleted': 0}
{'ids': ['id1', 'id2', 'id3'], 'embeddings': None, 'documents': ['doc1', 'doc2', 'doc3'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'chapter': '3', 'verse': '16'}, {'verse': '5', 'chapter': '3'}, {'chapter': '29', 'verse': '11'}]}


### 🗑️ Cell 31 — Same delete-then-verify pattern, different filter

Same idea as Cell 30, but filtering on a different metadata field (`verse: '5'`) — showing `delete()`'s `where` clause can filter on any metadata field, not just the one used previously.

In [ ]:
print(collection.delete(
    ids=["id1", "id2", "id3"],
	  where={"verse": "5"}
))

print(collection.get(
	ids=["id1", "id2", "id3"]
))

{'deleted': 1}
{'ids': ['id1', 'id3'], 'embeddings': None, 'documents': ['doc1', 'doc3'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'chapter': '3', 'verse': '16'}, {'chapter': '29', 'verse': '11'}]}
